# 21 — Bloom Filter (Amazon FAR style)

A probabilistic set: "definitely not present" or "probably present" — no false negatives, tunable
false positives, tiny memory. Parts 1-3 build the core (concurrency at Part 3); Parts 4-5 are stretch
tasks (deletable counting filter, then parallel build + union). Fill stubs, run each test cell, peek at
the collapsed `ref_` solutions only after trying.

`_hashes(item, k, size)` (provided) gives `k` **stable** bit positions (via `hashlib`, so they match
across processes).

---

## Part 1 — Bloom filter

`BloomFilter(size, num_hashes)` with `add(item)` and `contains(item)`. Set the `k` hashed bits on add;
`contains` is True only if **all** `k` bits are set.

**Lock down:** **no false negatives** (an added item always reports present); false *positives* are
possible (bits set by other items); membership is monotonic (adding never removes).

In [ ]:
import hashlib


def _hashes(item, k, size):
    return [int(hashlib.md5(("%d:%s" % (i, item)).encode()).hexdigest(), 16) % size for i in range(k)]


class BloomFilter:
    def __init__(self, size, num_hashes):
        raise NotImplementedError

    def add(self, item):
        raise NotImplementedError

    def contains(self, item):
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    bf = BloomFilter(size=1000, num_hashes=4)
    for x in ("apple", "banana", "cherry"):
        bf.add(x)
    assert all(bf.contains(x) for x in ("apple", "banana", "cherry"))   # no false negatives
    bf.add("durian"); assert bf.contains("durian")                      # monotonic
    print("Part 1 OK")

_t1()

## Part 2 — Optimal sizing

`optimal_params(n, p) -> (size, num_hashes)`: given expected item count `n` and target false-positive
rate `p`, compute `m = ceil(-n·ln p / (ln 2)²)` and `k = round((m/n)·ln 2)` (at least 1).

**Lock down:** these are the standard Bloom formulas; verify that a filter built with them hits roughly
the target FPR empirically (and still has zero false negatives).

In [ ]:
import math


def optimal_params(n, p):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    n, p = 1000, 0.01
    size, k = optimal_params(n, p)
    assert size > 0 and k >= 1
    bf = BloomFilter(size, k)
    for i in range(n):
        bf.add("item%d" % i)
    assert all(bf.contains("item%d" % i) for i in range(n))             # no false negatives
    fp = sum(bf.contains("other%d" % i) for i in range(5000))
    assert fp / 5000 < 0.05                                             # ~target 0.01, generous bound
    print("Part 2 OK")

_t2()

## Part 3 — Thread-safe Bloom filter

`ConcurrentBloom(size, num_hashes)`: thread-safe `add`/`contains`. Many threads add disjoint items at
once; afterwards **every** added item must still report present (no false negatives).

**Discuss:** setting a bit is idempotent so it's relatively benign, but bit-array mutation should still
be guarded; a read/write lock since reads dominate; sharded locks by bit region.

In [ ]:
import threading


class ConcurrentBloom:
    def __init__(self, size, num_hashes):
        raise NotImplementedError

    def add(self, item):
        raise NotImplementedError

    def contains(self, item):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    bf = ConcurrentBloom(20000, 5)

    def worker(t):
        for i in range(500):
            bf.add("t%d-i%d" % (t, i))

    ts = [threading.Thread(target=worker, args=(t,)) for t in range(8)]
    for t in ts: t.start()
    for t in ts: t.join()
    assert all(bf.contains("t%d-i%d" % (t, i)) for t in range(8) for i in range(500))   # no FN
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Counting Bloom filter (deletable)

A plain Bloom filter can't delete (clearing a bit might erase other items). `CountingBloom(size,
num_hashes)` uses per-slot **counters** instead of bits: `add` increments, `remove` decrements (only if
all slots are > 0, to avoid underflow), `contains` is True if all slots are > 0. Supports add/remove
with no false negatives.

**Discuss:** counter width / overflow, why removing a never-added item is unsafe in general (the
underflow guard), and the memory cost vs a plain bit filter.

In [ ]:
class CountingBloom:
    def __init__(self, size, num_hashes):
        raise NotImplementedError

    def add(self, item):
        raise NotImplementedError

    def remove(self, item):
        raise NotImplementedError

    def contains(self, item):
        raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    bf = CountingBloom(1000, 4)
    bf.add("a"); bf.add("a")                            # added twice
    bf.remove("a"); assert bf.contains("a")             # still present after one remove
    bf.remove("a"); assert not bf.contains("a")         # now gone
    bf.remove("a")                                      # underflow guard: no crash / no negatives
    assert not bf.contains("a") and all(c >= 0 for c in bf.counts)
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel build + union

`parallel_build(items, size, num_hashes, num_procs) -> bytearray`: build the filter's bit array over a
large item set in parallel with `ProcessPoolExecutor`, then **union** the per-chunk results (a Bloom
filter union is just OR of the bit arrays). Worker is `bloom_workers.build_bits` (returns set bit
positions for its chunk). Must equal the sequentially-built bit array.

**Discuss:** Bloom filters union losslessly via OR (same size/hashes); GIL (hashing is CPU-bound ->
processes); shipping chunks vs the whole set.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import bloom_workers


def parallel_build(items, size, num_hashes, num_procs) -> bytearray:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    items = ["x%d" % i for i in range(300)]
    size, k = 5000, 4
    bits = parallel_build(items, size, k, 4)
    assert all(all(bits[i] for i in _hashes(it, k, size)) for it in items)   # no false negatives
    seq = bytearray(size)
    for it in items:
        for idx in _hashes(it, k, size):
            seq[idx] = 1
    assert bits == seq                                  # equals the sequential union
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Scalable Bloom filters (grow as you add); partitioned Bloom; Cuckoo filter (deletes + better locality).
- Choosing k/m for a memory budget; measuring vs predicted FPR.
- Distributed membership: per-shard filters unioned at a coordinator.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import hashlib
import math
import threading
from concurrent.futures import ProcessPoolExecutor
import bloom_workers


def _hashes(item, k, size):
    return [int(hashlib.md5(("%d:%s" % (i, item)).encode()).hexdigest(), 16) % size for i in range(k)]


class RefBloomFilter:
    def __init__(self, size, num_hashes):
        self.size, self.k = size, num_hashes
        self.bits = bytearray(size)

    def add(self, item):
        for idx in _hashes(item, self.k, self.size):
            self.bits[idx] = 1

    def contains(self, item):
        return all(self.bits[idx] for idx in _hashes(item, self.k, self.size))


def ref_optimal_params(n, p):
    m = math.ceil(-n * math.log(p) / (math.log(2) ** 2))
    k = max(1, round((m / n) * math.log(2)))
    return m, k


class RefConcurrentBloom:
    def __init__(self, size, num_hashes):
        self.size, self.k = size, num_hashes
        self.bits = bytearray(size)
        self.lock = threading.Lock()

    def add(self, item):
        idxs = _hashes(item, self.k, self.size)
        with self.lock:
            for idx in idxs:
                self.bits[idx] = 1

    def contains(self, item):
        idxs = _hashes(item, self.k, self.size)
        with self.lock:
            return all(self.bits[idx] for idx in idxs)


class RefCountingBloom:
    def __init__(self, size, num_hashes):
        self.size, self.k = size, num_hashes
        self.counts = [0] * size

    def add(self, item):
        for idx in _hashes(item, self.k, self.size):
            self.counts[idx] += 1

    def remove(self, item):
        idxs = _hashes(item, self.k, self.size)
        if all(self.counts[idx] > 0 for idx in idxs):   # guard against underflow
            for idx in idxs:
                self.counts[idx] -= 1

    def contains(self, item):
        return all(self.counts[idx] > 0 for idx in _hashes(item, self.k, self.size))


def ref_parallel_build(items, size, num_hashes, num_procs):
    chunks = [items[i::num_procs] for i in range(num_procs)]
    bits = bytearray(size)
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        for s in ex.map(bloom_workers.build_bits, [(c, num_hashes, size) for c in chunks]):
            for idx in s:
                bits[idx] = 1
    return bits


_b = RefBloomFilter(500, 3); _b.add("a")
assert _b.contains("a")
_m, _k = ref_optimal_params(1000, 0.01); assert _m > 1000 and _k >= 1
_c = RefCountingBloom(200, 3); _c.add("z"); _c.remove("z"); assert not _c.contains("z")
_c.remove("z"); assert all(v >= 0 for v in _c.counts)
_items = ["q%d" % i for i in range(50)]
_seq = bytearray(1000)
for _it in _items:
    for _i in _hashes(_it, 3, 1000):
        _seq[_i] = 1
assert ref_parallel_build(_items, 1000, 3, 4) == _seq
print("reference solutions OK")